# SHAP global — régression logistique retenue

Complète l’explication **locale** de l’agent (`explain.py`, top 5 facteurs d’un client).

Ici : SHAP sur **tout le jeu de test** (même split que `train.py` : 80/20, `stratify`, `random_state=42`).

Les valeurs SHAP sont en **log-odds** de churn (après `StandardScaler`).

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config.settings import MODELS_DIR
from src.agents.prediction_agent.train import load_xy, split_train_test

FIGURES = ROOT / "notebooks" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

print("ROOT", ROOT)
print("MODELS_DIR", MODELS_DIR)

In [ ]:
model_path = MODELS_DIR / "selected_churn.joblib"
if not model_path.exists():
    model_path = MODELS_DIR / "logreg_churn.joblib"
model = joblib.load(model_path)

X, y = load_xy()
X_train, X_test, y_train, y_test = split_train_test(X, y)

scaler = model.named_steps["scaler"]
clf = model.named_steps["clf"]

bg_path = MODELS_DIR / "shap_background.joblib"
background = joblib.load(bg_path) if bg_path.exists() else X_train.sample(n=100, random_state=42)
background = background.reindex(columns=X_test.columns)

X_test_scaled = scaler.transform(X_test)
bg_scaled = scaler.transform(background)

print("model:", model_path.name)
print("test:", X_test.shape)
print("churn test:", round(float(y_test.mean()), 4))

In [ ]:
explainer = shap.LinearExplainer(clf, bg_scaled)
raw = explainer(X_test_scaled)
values = np.array(getattr(raw, "values", explainer.shap_values(X_test_scaled)), dtype=float)
if values.ndim == 3:
    values = values[:, :, 1]
base = getattr(raw, "base_values", explainer.expected_value)

explanation = shap.Explanation(
    values=values,
    base_values=base,
    data=np.array(X_test_scaled),
    feature_names=list(X_test.columns),
)
print("SHAP shape:", explanation.values.shape)

## Importance globale (moyenne |SHAP|)

In [ ]:
mean_abs = (
    pd.Series(np.abs(explanation.values).mean(axis=0), index=X_test.columns)
    .sort_values(ascending=False)
)
mean_abs.head(15)

In [ ]:
plt.figure(figsize=(8, 6))
shap.plots.bar(explanation, max_display=15, show=False)
plt.tight_layout()
plt.savefig(FIGURES / "shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()

## Beeswarm

Chaque point = un client du test. Couleur = valeur de la feature (rouge = élevé, bleu = bas).
À droite du 0 : la feature **augmente** le log-odds de churn.

In [ ]:
plt.figure(figsize=(8, 6))
shap.plots.beeswarm(explanation, max_display=15, show=False)
plt.tight_layout()
plt.savefig(FIGURES / "shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

## Waterfall (un client du test)

Même type d’explication que `persona.risk_factors`, mais détaillée.

In [ ]:
i = 0
print("y_test:", int(y_test.iloc[i]), "  proba:", round(float(model.predict_proba(X_test.iloc[[i]])[0, 1]), 4))
plt.figure(figsize=(8, 6))
shap.plots.waterfall(explanation[i], max_display=12, show=False)
plt.tight_layout()
plt.savefig(FIGURES / "shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()

## Interpretation for beesWarmplot

| ## Feature                 | ## Valeur élevée 🔴       | ## Valeur faible 🔵 |
| ----------------------- | ---------------------- | ---------------- |
| **tenure**              | ↓ churn                | ↑ churn          |
| **MonthlyCharges**      | ↓ churn dans ce modèle | ↑ churn          |
| **Fiber optic**         | ↑ churn                | ↓ churn          |
| **Contract Two year**   | ↓ churn                | ↑ churn          |
| **Contract One year**   | ↓ churn                | ↑ churn          |
| **StreamingMovies Yes** | ↑ churn                | ↓ churn          |
| **StreamingTV Yes**     | ↑ churn                | ↓ churn          |
| **MultipleLines Yes**   | ↑ churn                | ↓ churn          |
| **Electronic check**    | ↑ churn                | ↓ churn          |
| **PaperlessBilling**    | ↑ churn                | ↓ churn          |
| **TotalCharges**        | ↑ churn                | ↓ churn          |
| **OnlineSecurity Yes**  | ↓ churn                | ↑ churn          |


## Lecture (test, 1409 clients)

Importance moyenne |SHAP| :

1. `tenure` — ancienneté courte → plus de risque  
2. `MonthlyCharges` — facture élevée → plus de risque  
3. `InternetService_Fiber optic` — fibre → plus de risque  
4. `Contract_Two year` / `Contract_One year` — engagement → moins de risque  
5. Streaming / `Electronic check` / `PaperlessBilling` — signaux secondaires  

Les figures sont aussi enregistrées dans `notebooks/figures/` (`shap_bar.png`, `shap_beeswarm.png`, `shap_waterfall.png`).